# Network Simulation

iso8583sim can stand in for an issuer host, so you can test acquirer, switch or terminal software without a card network connection. This notebook runs everything in-process:

1. A **mock issuer host** that answers by rules
2. The **TCP client**, which matches responses to requests by STAN
3. A **load test**

The same things are available from the command line as `iso8583sim serve`, `send` and `load`.

In [1]:
import sys

sys.path.insert(0, "..")  # For development - remove in production

In [2]:
import asyncio
import logging

from iso8583sim.core.describe import describe_message
from iso8583sim.core.samples import sample_message
from iso8583sim.net import Framing, ISO8583Client, MockHost, Rule, run_load
from iso8583sim.wire import WireFormat

logging.disable(logging.INFO)  # keep the parser's per-message logs out of the output

## Start a mock host

Rules are checked in order and the first match wins. Here: large amounts are declined with `51`, one test BIN gets a slow `05`, another is never answered, and everything else is approved.

In [3]:
FORMAT = WireFormat.bcd()
FRAMING = Framing(header="2b", tpdu=bytes.fromhex("6000010000"))

host = MockHost(
    rules=[
        Rule(amount_gt=100_000, respond="51"),  # over 1,000.00: insufficient funds
        Rule(pan_prefix=["4000"], respond="05", delay_ms=300),
        Rule(pan_prefix=["4999"], action="drop"),  # never answer
        Rule(),  # approve everything else
    ],
    wire_format=FORMAT,
    framing=FRAMING,
)
await host.start("127.0.0.1", 0)  # port 0 picks a free port
print("Mock host listening on port", host.port)

Mock host listening on port 57238


## Send requests

In [4]:
client = ISO8583Client("127.0.0.1", host.port, wire_format=FORMAT, framing=FRAMING, timeout=2)

for request in [
    sample_message(stan="000001"),
    sample_message(amount_minor_units=250_000, stan="000002"),
    sample_message("echo", stan="000003"),
]:
    response = await client.send(request)
    print(describe_message(response)["summary"], "\n")

MTI 0110: authorization response from acquirer. Card network: VISA. Card: 411111******1111. Transaction type: Purchase. Amount: 10.00 USD. Response code 00: Approved. Approval code: A00001. Terminal: TERM0001. Merchant: MERCHANT123456. 

MTI 0110: authorization response from acquirer. Card network: VISA. Card: 411111******1111. Transaction type: Purchase. Amount: 2,500.00 USD. Response code 51: Insufficient funds. Terminal: TERM0001. Merchant: MERCHANT123456. 

MTI 0810: network management response from acquirer. Response code 00: Approved. Network management: Echo test. 



A dropped request raises `TimeoutError`, so you can test how your code handles an issuer that never answers:

In [5]:
try:
    await client.send(sample_message(pan="4999123412341234", stan="000004"), timeout=0.5)
except TimeoutError as e:
    print("TimeoutError:", e)

TimeoutError: No response to STAN 000004 within 0.5s


## Responses in any order

The slow `05` rule answers after 300 ms. A request sent after it still gets its own response first, because the client matches on STAN:

In [6]:
import time

start = time.perf_counter()
slow = asyncio.create_task(client.send(sample_message(pan="4000123412341234", stan="000005")))
await asyncio.sleep(0.05)
fast = await client.send(sample_message(stan="000006"))
print(f"fast (STAN {fast.fields[11]}): code {fast.fields[39]} after {(time.perf_counter() - start) * 1000:.0f} ms")
slow = await slow
print(f"slow (STAN {slow.fields[11]}): code {slow.fields[39]} after {(time.perf_counter() - start) * 1000:.0f} ms")
await client.close()

fast (STAN 000006): code 00 after 52 ms


slow (STAN 000005): code 05 after 303 ms


## Load test

`run_load` sends generated requests with a fixed number in flight and reports throughput and latency. Client and host share this notebook's event loop, so treat the numbers as a floor.

In [7]:
report = await run_load(
    lambda: ISO8583Client("127.0.0.1", host.port, wire_format=FORMAT, framing=FRAMING),
    count=5_000,
    concurrency=50,
    connections=2,
)
print(f"responses {report.responses:,} of {report.sent:,}, timeouts {report.timeouts}, errors {report.errors}")
print(f"throughput {report.throughput:,.0f} msg/s")
print(f"latency p50 {report.percentile(50):.2f} ms, p99 {report.percentile(99):.2f} ms")
print("response codes:", dict(report.response_codes))

responses 5,000 of 5,000, timeouts 0, errors 0
throughput 11,142 msg/s
latency p50 4.36 ms, p99 7.27 ms
response codes: {'00': 5000}


In [8]:
print(host.stats)
await host.stop()

HostStats(received=5006, responded=5005, dropped=1, errors=0, response_codes=Counter({'00': 5003, '51': 1, '05': 1}))


## From the command line

```bash
# terminal 1
iso8583sim serve --port 8583 --rules rules.yaml --format bcd --tpdu 6000010000

# terminal 2
iso8583sim generate --type auth -o msg.txt
iso8583sim send "$(cat msg.txt)" --port 8583 --format bcd --tpdu 6000010000
iso8583sim load --port 8583 --format bcd --tpdu 6000010000 --count 10000 --concurrency 50
```

See the [Networking docs](https://iso8583sim.com/docs/net) for the rules file format.